# 9 - Embeddings no LangChain

**Embeddings** são representações numéricas de textos — vetores de números decimais que capturam o *significado semântico* de uma frase ou documento.

A ideia central: textos com significados parecidos geram vetores próximos no espaço vetorial.

```
"Eu gosto de cachorro"  →  [0.12, -0.34, 0.87, ...]  ──┐
"Eu gosto de animais"   →  [0.11, -0.31, 0.85, ...]  ──┤ → próximos (similar)
"Hoje está chovendo"    →  [-0.45, 0.22, -0.10, ...] ──┘ → distante dos outros
```

Isso é a base de sistemas de **busca semântica**, **RAG (Retrieval-Augmented Generation)** e **memória de agentes**.

### Pacotes necessários

```bash
pip install langchain-openai
```

> **Import moderno**: o modelo de embeddings está em `langchain_openai` (pacote separado `langchain-openai`).
> Em versões antigas era `from langchain.embeddings import OpenAIEmbeddings` — isso está **deprecado**.

## 1. Criando o modelo de embeddings

O `OpenAIEmbeddings` usa por padrão o modelo `text-embedding-3-small` (mais rápido e barato).
Para maior precisão, use `model="text-embedding-3-large"`.

| Modelo | Dimensões | Uso recomendado |
|--------|-----------|------------------|
| `text-embedding-3-small` | 1536 | Padrão — boa relação custo/qualidade |
| `text-embedding-3-large` | 3072 | Máxima precisão semântica |
| `text-embedding-ada-002` | 1536 | Legado — evite em novos projetos |

In [2]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

# Modelo padrão: text-embedding-3-small (1536 dimensões)
embedding_model = OpenAIEmbeddings()

# Para usar o modelo maior (3072 dimensões):
# embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

## 2. Embeddings de documentos — `embed_documents()`

Use `embed_documents()` quando quiser vetorizar **múltiplos textos de uma vez** (ex: para indexar em um banco vetorial).

Retorna uma **lista de vetores**, um para cada texto.

In [3]:
textos = [
    "Eu gosto de cachorro",
    "Eu gosto de animais",
    "Hoje a tarde está chovendo",
]

embeddings = embedding_model.embed_documents(textos)

print(f"Número de embeddings gerados: {len(embeddings)}")
print(f"Dimensões de cada vetor: {len(embeddings[0])}")

Número de embeddings gerados: 3
Dimensões de cada vetor: 1536


## 3. Medindo similaridade com produto escalar

A forma mais comum de comparar dois vetores é o **produto escalar** (dot product) ou a **similaridade de cosseno**.

Quando os vetores já são normalizados (como os da OpenAI), os dois são equivalentes.

- Valor próximo de **1.0** → textos muito similares
- Valor próximo de **0.0** → textos sem relação semântica

> A OpenAI normaliza os vetores por padrão, então podemos usar `np.dot()` diretamente como proxy de similaridade.

In [5]:
import numpy as np

# Comparando: "cachorro" vs "animais" (devem ser similares)
sim_cachorro_animais = np.dot(embeddings[0], embeddings[1])
print(f"Similaridade 'cachorro' × 'animais':  {sim_cachorro_animais:.4f}")

# Comparando: "cachorro" vs "chovendo" (devem ser diferentes)
sim_cachorro_chuva = np.dot(embeddings[0], embeddings[2])
print(f"Similaridade 'cachorro' × 'chovendo': {sim_cachorro_chuva:.4f}")

# Comparando: "animais" vs "chovendo"
sim_animais_chuva = np.dot(embeddings[1], embeddings[2])
print(f"Similaridade 'animais'  × 'chovendo': {sim_animais_chuva:.4f}")

Similaridade 'cachorro' × 'animais':  0.9383
Similaridade 'cachorro' × 'chovendo': 0.8206
Similaridade 'animais'  × 'chovendo': 0.8070


## 4. Matriz de similaridade completa

Para visualizar todas as combinações de uma vez, podemos construir uma **matriz de similaridade**.

A diagonal principal sempre vale `1.0` (cada texto comparado consigo mesmo).

In [6]:
labels = ["cachorro", "animais", "chovendo"]

print(f"{'':>12}", end="")
for label in labels:
    print(f"{label:>12}", end="")
print()

for i in range(len(embeddings)):
    print(f"{labels[i]:>12}", end="")
    for j in range(len(embeddings)):
        sim = round(np.dot(embeddings[i], embeddings[j]), 2)
        print(f"{sim:>12}", end="")
    print()

                cachorro     animais    chovendo
    cachorro         1.0        0.94        0.82
     animais        0.94         1.0        0.81
    chovendo        0.82        0.81         1.0


## 5. Embedding de query — `embed_query()`

Use `embed_query()` quando quiser vetorizar **uma única frase de busca** (ex: a pergunta do usuário).

A distinção entre `embed_documents()` e `embed_query()` existe porque alguns provedores usam
instruções internas diferentes para otimizar cada caso de uso.

> **Regra prática**: documentos para indexar → `embed_documents()`. Pergunta do usuário → `embed_query()`.

In [7]:
pergunta = "O que é um cachorro?"
emb_query = embedding_model.embed_query(pergunta)

print(f"Dimensões do vetor: {len(emb_query)}")
print(f"Primeiros 10 valores: {emb_query[:10]}")

Dimensões do vetor: 1536
Primeiros 10 valores: [0.005100993439555168, 0.003876505186781287, -0.0046824184246361256, -0.006441058591008186, -0.017842544242739677, 0.013619309291243553, 0.0015954270493239164, -0.002005412010475993, -0.002075695199891925, 0.005591413471847773]


## 6. Busca semântica manual

Com `embed_query()` e `embed_documents()` podemos implementar uma **busca semântica simples**:
dado uma pergunta, encontrar qual dos documentos é mais relevante.

In [9]:
# Calculando similaridade da pergunta com cada documento
print(f"Pergunta: '{pergunta}'\n")

similaridades = []
for i, texto in enumerate(textos):
    sim = np.dot(emb_query, embeddings[i])
    similaridades.append((sim, texto))
    print(f"  [{sim:.4f}] {texto}")

# Documento mais relevante
melhor = max(similaridades, key=lambda x: x[0])
print(f"\n→ Documento mais relevante: '{melhor[1]}' (score: {melhor[0]:.4f})")

Pergunta: 'O que é um cachorro?'

  [0.8824] Eu gosto de cachorro
  [0.8303] Eu gosto de animais
  [0.7812] Hoje a tarde está chovendo

→ Documento mais relevante: 'Eu gosto de cachorro' (score: 0.8824)


## 7. Usando outros provedores de embeddings

O LangChain suporta muitos provedores além da OpenAI. A interface é sempre a mesma:
`embed_documents()` e `embed_query()`.

```python
# Google (Vertex AI)
from langchain_google_vertexai import VertexAIEmbeddings
model = VertexAIEmbeddings(model="text-embedding-004")

# Cohere
from langchain_cohere import CohereEmbeddings
model = CohereEmbeddings(model="embed-multilingual-v3.0")

# Modelos locais via Ollama (sem custo de API)
from langchain_ollama import OllamaEmbeddings
model = OllamaEmbeddings(model="nomic-embed-text")

# HuggingFace (modelos open-source)
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
```

> **Dica**: para produção em português, considere `text-embedding-3-large` (OpenAI) ou
> `embed-multilingual-v3.0` (Cohere), que têm excelente suporte multilingual.

## Resumo

| Método | Quando usar |
|--------|-------------|
| `embed_documents(lista)` | Vetorizar múltiplos textos para indexação |
| `embed_query(texto)` | Vetorizar a pergunta/busca do usuário |
| `np.dot(v1, v2)` | Medir similaridade entre dois vetores normalizados |

**Fluxo típico em um sistema RAG:**

```
Documentos  →  embed_documents()  →  Vector Store (indexação)
                                              ↑
Pergunta    →  embed_query()      →  Busca por similaridade  →  LLM  →  Resposta
```

> **Próximo passo**: unir text splitting + embeddings em um **Vector Store** (ex: FAISS, Chroma)
> para criar um sistema completo de busca semântica sobre documentos.